<a href="https://colab.research.google.com/github/BakrAdli/My_AI_Journey/blob/main/smart_car_diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import json
import os

# ==========================================
# V3.0: LLM / API INITIALIZATION (DYNAMIC AUTO-SELECT)
# ==========================================
try:
    import google.generativeai as genai
    from google.colab import userdata

    # 1. Fetch the secure API key
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)

    # 2. Dynamic Model Selection (Find active generateContent model)
    supported_models = [
        m.name for m in genai.list_models()
        if 'generateContent' in m.supported_generation_methods
    ]

    # Use gemini-3-flash-preview if present, otherwise pick the first valid model
    chosen_model = 'gemini-3-flash-preview'
    if chosen_model not in [m.replace('models/', '') for m in supported_models]:
        chosen_model = supported_models[0] if supported_models else 'gemini-3-flash-preview'

    model = genai.GenerativeModel(chosen_model)
    LLM_READY = True
    print(f" [System] Connected successfully to AI Model: {chosen_model}")

except Exception as e:
    LLM_READY = False
    print(f" [Warning] AI features disabled. API Setup error: {e}")
# ==========================================
# V3.0: AI ASSISTANT CLASS
# ==========================================
class FleetAI:
    def __init__(self, model):
        self.model = model
        self.system_prompt = (
            "You are a professional AI Fleet Mechanic. "
            "Analyze the provided JSON garage data and answer the user's questions "
            "concisely with expert maintenance advice. Base your answers ONLY on the provided data."
        )

    def ask_mechanic(self, fleet_data: str, user_question: str) -> str:
        if not LLM_READY:
            return " AI is currently offline. Please check your Colab Secrets."

        # 3. Context Injection: Combine instructions, live data, and the user's question
        prompt = f"{self.system_prompt}\n\nFleet Data:\n{fleet_data}\n\nUser Question: {user_question}"

        try:
            # 5. Error Handling & Generation
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            return f" AI Communication Error: {e}"

# ==========================================
# CORE SYSTEM: CAR & GARAGE CLASSES
# ==========================================
class Car:
    OIL_INTERVAL = 10000
    TIRE_INTERVAL = 50000

    def __init__(self, brand, model, engine_type, mileage=0):
        self.brand = brand
        self.model = model
        self.engine_type = engine_type
        self.mileage = mileage
        self.oil_change_at = mileage
        self.tire_change_at = mileage

    def add_trip(self, distance):
        self.mileage += distance

    def reset_oil(self):
        self.oil_change_at = self.mileage

    def reset_tires(self):
        self.tire_change_at = self.mileage

    def get_oil_status(self) -> str:
        driven = self.mileage - self.oil_change_at
        remaining = self.OIL_INTERVAL - driven
        if remaining <= 0:
            return f" CHANGE_REQUIRED ({driven:,} km driven)"
        return f" OK ({remaining:,} km left)"

    def get_tires_status(self) -> str:
        driven = self.mileage - self.tire_change_at
        remaining = self.TIRE_INTERVAL - driven
        if remaining <= 0:
            return f" CHECK_TREAD ({driven:,} km driven)"
        return f" OK ({remaining:,} km left)"

    def get_engine_tip(self) -> str:
        tips = {
            "GASOLINE": "Check spark plugs & fuel injectors",
            "DIESEL": "Check Glow Plugs & EGR Valve",
            "HYBRID": "Schedule battery health diagnostic",
            "ELECTRIC": "Inspect thermal management & brake regen"
        }
        return tips.get(self.engine_type.upper(), "Standard engine maintenance")

    def get_llm_context(self) -> dict:
        return {
            "vehicle": f"{self.brand} {self.model}",
            "specs": {"engine": self.engine_type, "mileage": self.mileage},
            "health": {
                "oil": self.get_oil_status(),
                "tires": self.get_tires_status(),
                "engine": self.get_engine_tip()
            }
        }

    def to_dict(self):
        return {
            "brand": self.brand,
            "model": self.model,
            "engine_type": self.engine_type,
            "mileage": self.mileage,
            "oil_change_at": self.oil_change_at,
            "tire_change_at": self.tire_change_at
        }

    @classmethod
    def from_dict(cls, data):
        car = cls(data['brand'], data['model'], data['engine_type'], data['mileage'])
        car.oil_change_at = data.get('oil_change_at', data['mileage'])
        car.tire_change_at = data.get('tire_change_at', data['mileage'])
        return car

class Garage:
    def __init__(self):
        self.cars = []
        self.active_index = 0

    def add_car(self, car):
        self.cars.append(car)
        self.active_index = len(self.cars) - 1

    def get_active_car(self):
        if not self.cars:
            return None
        return self.cars[self.active_index]

    def print_active_banner(self):
        car = self.get_active_car()
        if not car:
            print(" [*] No vehicles found in Garage.")
            return
        print("═══════════════════════════════════════════════════════")
        print(f"   ACTIVE VEHICLE: {car.brand} {car.model}")
        print(f"  • Engine : {car.engine_type} | Mileage: {car.mileage:,} km")
        print(f"  • Status : Oil [{car.get_oil_status()}] | Tires [{car.get_tires_status()}]")
        print("═══════════════════════════════════════════════════════")

    def show_maintenance_dashboard(self):
        print("\n═══════════════════════════════════════════════════════")
        print("  FLEET MAINTENANCE DASHBOARD (HEALTH OVERVIEW)")
        print("═══════════════════════════════════════════════════════")
        for idx, car in enumerate(self.cars):
            active_tag = " [ACTIVE]" if idx == self.active_index else ""
            print(f"\n [{idx}] {car.brand} {car.model} ({car.mileage:,} km){active_tag}")
            print(f"   ├─ Oil Status  : {car.get_oil_status()}")
            print(f"   ├─ Tire Status : {car.get_tires_status()}")
            print(f"   └─ Engine Tip  : {car.get_engine_tip()}")
        print("═══════════════════════════════════════════════════════\n")

    def get_fleet_llm_context(self) -> str:
        context = []
        for idx, car in enumerate(self.cars):
            context.append({
                "garage_id": idx,
                "is_active_vehicle": (idx == self.active_index),
                "car_data": car.get_llm_context()
            })
        return json.dumps(context, indent=2)

    def save_fleet(self, filename="fleet_data.json"):
        with open(filename, "w") as f:
            json.dump([car.to_dict() for car in self.cars], f, indent=4)
        print(f"   [Storage] Saved to '{filename}'. (Click ' Refresh' in Colab if hidden)")

    def load_fleet(self, filename="fleet_data.json"):
        if os.path.exists(filename):
            with open(filename, "r") as f:
                data = json.load(f)
                self.cars = [Car.from_dict(d) for d in data]
            return True
        return False

# ==========================================
# MAIN MENU & INTERFACE
# ==========================================
def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')

def main():
    garage = Garage()
    garage.load_fleet()

    # Initialize the AI Mechanic if the API key was valid
    fleet_ai = FleetAI(model) if LLM_READY else None

    while True:
        clear_screen()
        print("\n=======================================================")
        print("== Smart Diagnostics Initialized ==")
        print("=======================================================\n")

        garage.print_active_banner()

        print(f"\n==================================================")
        print(f" GARAGE MANAGMENT SYSTEM V3.0 (Fleet: {len(garage.cars)})")
        print(f"==================================================")

        print("\n[Menu Options]:")
        print(" 1. Add Trip")
        print(" 2. Reset Oil")
        print(" 3. Reset Tires")
        print(" 4. Edit Vehicle (Under Construction)")
        print(" 5. Garage Fleet Manager")
        print(" 6. Fleet Health Dashboard")
        print(" 7. Ask AI Fleet Mechanic  (NEW)")
        print(" 8. View AI Context")
        print(" 9. Exit")

        choice = input("\nSelect (1-9): ")
        car = garage.get_active_car()

        if choice == '1' and car:
            dist = int(input("Trip distance (KM): "))
            car.add_trip(dist)
            garage.save_fleet()
            print(f"  [System] Trip recorded! Odometer: {car.mileage:,} km")
            input("\nPress Enter...")

        elif choice == '2' and car:
            car.reset_oil()
            garage.save_fleet()
            print("  [System] Oil reset successful.")
            input("\nPress Enter...")

        elif choice == '3' and car:
            car.reset_tires()
            garage.save_fleet()
            print("  [System] Tires reset successful.")
            input("\nPress Enter...")

        elif choice == '5':
            print("\n --- GARAGE FLEET MANAGER ---")
            for idx, c in enumerate(garage.cars):
                status = "[ACTIVE]" if idx == garage.active_index else ""
                print(f"   [{idx}] {c.brand} {c.model} ({c.engine_type}) - {c.mileage:,} km  {status}")

            action = input("\nActions: [S]witch Active | [A]dd New Car | [R]eturn\nSelect Action (S/A/R): ").upper()
            if action == 'S':
                idx = int(input("Enter vehicle ID to set active: "))
                if 0 <= idx < len(garage.cars):
                    garage.active_index = idx
                    garage.save_fleet()
            elif action == 'A':
                b = input(" Brand : ")
                m = input(" Model : ")
                e = input(" Engine : ")
                mil = int(input(" Mileage : "))
                garage.add_car(Car(b, m, e, mil))
                garage.save_fleet()
                print(f" [Garage] added {b} {m} to fleet and set as Active!")
                input("\nPress Enter...")

        elif choice == '6':
            garage.show_maintenance_dashboard()
            input("\nPress Enter...")

        # 4. Integrating the AI into the Main Menu
        elif choice == '7':
            if not LLM_READY or not fleet_ai:
                print("\n AI is offline. Make sure you set your GEMINI_API_KEY in Colab Secrets.")
            else:
                print("\n --- AI FLEET MECHANIC ---")
                q = input("Ask the mechanic (e.g., 'Does my Hyundai need maintenance?'): ")
                print("\nThinking...\n")
                answer = fleet_ai.ask_mechanic(garage.get_fleet_llm_context(), q)
                print(f" Mechanic Says:\n\n{answer}\n")
                print("-" * 55)
            input("Press Enter to continue...")

        elif choice == '8':
            print("\n[AI JSON Context Preview]:")
            print(garage.get_fleet_llm_context())
            input("\nPress Enter...")

        elif choice == '9':
            print("Exiting... Safe travels!")
            break
        else:
            input("Invalid choice or no car. Press Enter...")

if __name__ == "__main__":

    main()

 [System] Connected successfully to AI Model: gemini-3-flash-preview

== Smart Diagnostics Initialized ==

═══════════════════════════════════════════════════════
   ACTIVE VEHICLE: Toyota RAV4
  • Engine : GASOLINE | Mileage: 15,000 km
  • Status : Oil [ OK (10,000 km left)] | Tires [ OK (50,000 km left)]
═══════════════════════════════════════════════════════

 GARAGE MANAGMENT SYSTEM V3.0 (Fleet: 1)

[Menu Options]:
 1. Add Trip
 2. Reset Oil
 3. Reset Tires
 4. Edit Vehicle (Under Construction)
 5. Garage Fleet Manager
 6. Fleet Health Dashboard
 7. Ask AI Fleet Mechanic  (NEW)
 8. View AI Context
 9. Exit

Select (1-9): 7

 --- AI FLEET MECHANIC ---
Ask the mechanic (e.g., 'Does my Hyundai need maintenance?'): What is the health status of my Toyota?

Thinking...

 Mechanic Says:

Your Toyota RAV4 is in generally good health, but requires specific engine maintenance. 

**Health Summary:**
*   **Oil:** OK (10,000 km remaining).
*   **Tires:** OK (50,000 km remaining).
*   **Engine:*